In [ ]:
# Phase 3: Quality Control + Classifier Training

This notebook covers:
1. Evaluating GAN-generated image quality
2. Filtering synthetic images
3. Training EfficientNet-B2 classifier on augmented dataset

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.config import load_config, create_directories
from src.utils.logger import setup_logger

config = load_config('../config.yaml')
create_directories(config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## Step 1: Generate Synthetic Images from Trained GAN

In [ ]:
from src.train_gan import GANTrainer
from src.data.mvtec_dataset import MVTecDataLoader

# Load trained GAN
trainer = GANTrainer(config, device=device)

# Find latest checkpoint
ckpt_dir = Path(config.training.checkpoint_dir)
checkpoints = sorted(ckpt_dir.glob('checkpoint_epoch_*.pt'))

if checkpoints:
    latest_ckpt = checkpoints[-1]
    epoch = trainer.load_checkpoint(str(latest_ckpt))
    print(f'Loaded checkpoint from epoch {epoch}: {latest_ckpt}')
else:
    print('No checkpoints found. Run train_gan.py first.')

In [ ]:
# Generate synthetic images for each category
synthetic_dir = Path(config.training.output_dir) / 'synthetic'

for category in config.data.categories[:3]:  # Start with first 3 categories
    print(f'Generating for: {category}')
    
    loader = MVTecDataLoader(
        root_dir=config.data.raw_dir,
        category=category,
        batch_size=8,
        num_workers=0,
        image_size=config.data.image_size,
    )
    train_loader, _, _ = loader.get_loaders()
    
    out_dir = synthetic_dir / category
    trainer.generate_synthetic_images(train_loader, str(out_dir), num_images=100)
    print(f'  Saved to {out_dir}')

## Step 2: Quality Evaluation

In [ ]:
from src.evaluate_quality import QualityEvaluator

evaluator = QualityEvaluator(config, device=device)

category = config.data.categories[0]
synthetic_path = str(synthetic_dir / category)
real_path = str(Path(config.data.raw_dir) / category / 'train' / 'good')
filtered_path = str(Path(config.training.output_dir) / 'filtered' / category)

df_scores = evaluator.filter_synthetic_images(
    synthetic_dir=synthetic_path,
    real_dir=real_path,
    output_dir=filtered_path,
    keep_ratio=config.quality_control.keep_ratio,
)

print(df_scores[['image', 'fid', 'lpips', 'coverage', 'sharpness', 'final']].head(10))

In [ ]:
# Visualize quality distribution
evaluator.visualize_quality_distribution(
    df_scores,
    output_path=str(Path(config.training.output_dir) / 'quality_distribution.png')
)

# Print report
report = evaluator.generate_quality_report(
    df_scores,
    output_path=str(Path(config.training.output_dir) / 'quality_report.txt')
)
print(report)

## Step 3: Train Classifier

In [ ]:
from src.train_classifier import ClassifierTrainer

clf_trainer = ClassifierTrainer(config, device=device)

category = config.data.categories[0]
loader = MVTecDataLoader(
    root_dir=config.data.raw_dir,
    category=category,
    batch_size=config.data.batch_size,
    num_workers=0,
    image_size=config.data.image_size,
)
train_loader, val_loader, test_loader = loader.get_loaders()

print(f'Training classifier on: {category}')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Train (run full training from CLI: python src/train_classifier.py --config config.yaml)
clf_trainer.train(train_loader, val_loader)

## Step 4: Evaluate on Test Set

In [ ]:
test_metrics = clf_trainer.validate(test_loader)

print('\n=== TEST RESULTS ===')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')

## Step 5: Run Inference on a Single Image

In [ ]:
from src.inference import preprocess_image, classify_image

# Replace with an actual image path from your dataset
test_image_path = f"{config.data.raw_dir}/{category}/test/good/000.png"

image_tensor = preprocess_image(test_image_path, config.data.image_size)
result = classify_image(clf_trainer.model, image_tensor, config.data.categories, device)

print(f'Image      : {test_image_path}')
print(f'Prediction : {result["predicted_class"]}')
print(f'Confidence : {result["confidence"]:.4f}')
print(f'Defective  : {result["is_defective"]}')